In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 51.50853,
	"longitude": -0.12574,
	"hourly": ["precipitation", "wind_speed_10m", "wind_gusts_10m", "relative_humidity_2m", "temperature_2m", "surface_pressure"],
	"timezone": "Europe/London",
	"start_date": "2026-01-06",
	"end_date": "2026-01-06",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_precipitation = hourly.Variables(0).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(1).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(2).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(3).ValuesAsNumpy()
hourly_temperature_2m = hourly.Variables(4).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
hourly_data["surface_pressure"] = hourly_surface_pressure

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 51.5°N -0.12000012397766113°E
Elevation: 23.0 m asl
Timezone: b'Europe/London'None
Timezone difference to GMT+0: 0s

Hourly data
                         date  temperature_2m  precipitation  \
0  2026-01-06 00:00:00+00:00          -0.715            0.0   
1  2026-01-06 01:00:00+00:00          -1.065            0.0   
2  2026-01-06 02:00:00+00:00          -1.515            0.0   
3  2026-01-06 03:00:00+00:00          -1.965            0.0   
4  2026-01-06 04:00:00+00:00          -2.215            0.0   
5  2026-01-06 05:00:00+00:00          -2.315            0.0   
6  2026-01-06 06:00:00+00:00          -2.315            0.0   
7  2026-01-06 07:00:00+00:00          -2.265            0.0   
8  2026-01-06 08:00:00+00:00          -2.165            0.0   
9  2026-01-06 09:00:00+00:00          -2.365            0.0   
10 2026-01-06 10:00:00+00:00          -1.715            0.0   
11 2026-01-06 11:00:00+00:00          -0.715            0.0   
12 2026-01-06 12:00:00+00:00          

In [3]:
hourly_dataframe

,date,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,surface_pressure
0,2026-01-06 00:00:00+00:00,-0.715,0.0,77.0,5.506941,11.159999,1015.168579
1,2026-01-06 01:00:00+00:00,-1.065,0.0,80.0,6.618519,14.040000,1015.962830
2,2026-01-06 02:00:00+00:00,-1.515,0.0,83.0,7.862518,16.199999,1016.156982
3,2026-01-06 03:00:00+00:00,-1.965,0.0,86.0,7.289444,15.480000,1015.853149
4,2026-01-06 04:00:00+00:00,-2.215,0.0,87.0,7.091177,15.119999,1015.451843
5,2026-01-06 05:00:00+00:00,-2.315,0.0,89.0,7.628263,15.480000,1015.350769
6,2026-01-06 06:00:00+00:00,-2.315,0.0,91.0,8.089993,15.480000,1015.251038
7,2026-01-06 07:00:00+00:00,-2.265,0.0,89.0,8.225035,18.000000,1015.052246
8,2026-01-06 08:00:00+00:00,-2.165,0.0,88.0,8.373386,17.639999,1015.452087
9,2026-01-06 09:00:00+00:00,-2.365,0.0,89.0,7.729527,17.280001,1015.449890


In [ ]:
summary = hourly_dataframe[[
    'temperature_2m', 
    'precipitation', 
    'relative_humidity_2m', 
    'wind_speed_10m', 
    'wind_gusts_10m', 
    'surface_pressure']].mean()



temperature_2m             0.178750
precipitation              0.062500
relative_humidity_2m      84.458336
wind_speed_10m             9.573752
wind_gusts_10m            19.920000
surface_pressure        1012.751648
dtype: float32

In [ ]:
summary.to_dict()

{'temperature_2m': 0.17874999344348907,
 'precipitation': 0.0625000074505806,
 'relative_humidity_2m': 84.45833587646484,
 'wind_speed_10m': 9.573752403259277,
 'wind_gusts_10m': 19.920000076293945,
 'surface_pressure': 1012.7516479492188}